In [2]:
def suffix_array_to_bwt(suffix_array, text):
    n = len(text)
    result = bytearray()
    for pos in suffix_array:
        prev_pos = (pos - 1) % n
        result.append(text[prev_pos])
    
    return bytes(result)

In [15]:
def suffix_array_to_bwt(suffix_array, text):
    n = len(text)
    result = bytearray()
    for pos in suffix_array:
        prev_pos = (pos - 1) % n
        val = text[prev_pos]
        if isinstance(val, str):
            val = ord(val)
        result.append(val)
    return bytes(result)

def build_suffix_array(text):
    n = len(text)
    if n == 0: return []
    
    # Приводим к числам для удобства
    nums = [ord(c) for c in text] if isinstance(text, str) else list(text)
    
    sa = list(range(n))
    # Начальная сортировка по первому символу
    sa.sort(key=lambda i: nums[i])
    
    rank = [0] * n
    rank[sa[0]] = 0
    for i in range(1, n):
        rank[sa[i]] = rank[sa[i-1]] + (1 if nums[sa[i]] != nums[sa[i-1]] else 0)
            
    k = 1
    while k < n:
        # Сортировка по парам рангов (текущий, следующий через k)
        sa.sort(key=lambda i: (rank[i], rank[(i + k) % n]))
        
        new_rank = [0] * n
        new_rank[sa[0]] = 0
        for i in range(1, n):
            prev, curr = sa[i-1], sa[i]
            pair_prev = (rank[prev], rank[(prev + k) % n])
            pair_curr = (rank[curr], rank[(curr + k) % n])
            new_rank[curr] = new_rank[prev] + (1 if pair_prev != pair_curr else 0)
        
        rank = new_rank
        if rank[sa[n-1]] == n - 1: break # Все ранги уникальны
        k *= 2
        
    return sa

def bwt_transform(text):
    # Основная функция преобразования.
    if isinstance(text, bytes):
        text = text.decode('utf-8')
    sa = build_suffix_array(text)
    return suffix_array_to_bwt(sa, text)

s = "banana"
res = bwt_transform(s)
print(f"Text: {s}")
print(f"BWT:  {res.decode('utf-8')}")

Text: banana
BWT:  nnbaaa


In [6]:
def lz77_encode(data: bytes, window_size: int = 4096, lookahead_size: int = 18):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    while pos < n:
        # Определяем границы буфера поиска
        search_start = max(0, pos - window_size)
        search_end = pos
        search_buffer = data[search_start:search_end]
        
        # Ищем максимальное совпадение
        best_offset = 0
        best_length = 0
        
        # Максимальная длина совпадения ограничена буфером просмотра
        max_lookahead = min(lookahead_size, n - pos)
        
        # Перебираем возможные длины совпадения
        for length in range(1, max_lookahead + 1):
            # Ищем подстроку data[pos:pos+length] в буфере поиска
            pattern = data[pos:pos + length]
            
            # Поиск справа налево для нахождения ближайшего совпадения
            offset = search_buffer.rfind(pattern)
            
            if offset != -1:
                # Найдено совпадение
                # offset отсчитывается от конца search_buffer
                actual_offset = len(search_buffer) - offset
                if actual_offset > best_length or length > best_length:
                    best_offset = actual_offset
                    best_length = length
            else:
                break
        
        # Определяем следующий символ
        next_pos = pos + best_length
        next_byte = data[next_pos] if next_pos < n else 0
        
        # Кодируем тройку (смещение, длина, символ)
        # Используем 2 байта для смещения и 2 байта для длины
        encoded.extend(best_offset.to_bytes(2, 'big'))
        encoded.extend(best_length.to_bytes(2, 'big'))
        encoded.append(next_byte)
        
        # Сдвигаем окно
        pos += best_length + 1
    
    return bytes(encoded)


def lz77_decode(encoded: bytes):
    decoded = bytearray()
    i = 0
    n = len(encoded)
    
    while i < n:
        # Читаем тройку: смещение (2 байта), длина (2 байта), символ (1 байт)
        if i + 5 > n:
            break
            
        offset = int.from_bytes(encoded[i:i+2], 'big')
        length = int.from_bytes(encoded[i+2:i+4], 'big')
        next_byte = encoded[i+4]
        i += 5
        
        # Копируем совпадение из уже декодированных данных
        if length > 0:
            start_pos = len(decoded) - offset
            for j in range(length):
                decoded.append(decoded[start_pos + j])
        
        # Добавляем следующий символ
        decoded.append(next_byte)
    
    return bytes(decoded)

print("OK" if lz77_decode(lz77_encode(b"banana")) == b"banana" else "FAIL")

OK


In [8]:
def lzss_encode(data: bytes, window_size: int = 4096, lookahead_size: int = 18, min_match: int = 3):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    while pos < n:
        search_start = max(0, pos - window_size)
        search_buffer = data[search_start:pos]
        max_lookahead = min(lookahead_size, n - pos)
        
        best_offset = 0
        best_length = 0
        
        for length in range(min_match, max_lookahead + 1):
            pattern = data[pos:pos + length]
            offset = search_buffer.rfind(pattern)
            
            if offset != -1:
                actual_offset = len(search_buffer) - offset
                if actual_offset <= window_size:
                    best_offset = actual_offset
                    best_length = length
            else:
                break
        
        if best_length >= min_match:
            # Флаг 0x80 + смещение (2 байта) + длина (1 байт)
            encoded.append(0x80 | ((best_offset >> 8) & 0x7F))
            encoded.append(best_offset & 0xFF)
            encoded.append(best_length)
            pos += best_length
        else:
            # Флаг 0x00 + литерал
            encoded.append(0x00)
            encoded.append(data[pos])
            pos += 1
    
    return bytes(encoded)


def lzss_decode(encoded: bytes):
    decoded = bytearray()
    i = 0
    n = len(encoded)
    
    while i < n:
        flag = encoded[i]
        i += 1
        
        if flag == 0:
            # Литерал
            if i >= n:
                break
            decoded.append(encoded[i])
            i += 1
        else:
            # Ссылка
            if i + 2 > n:
                break
            offset = ((flag & 0x7F) << 8) | encoded[i]
            length = encoded[i + 1]
            i += 2
            
            start_pos = len(decoded) - offset
            for j in range(length):
                decoded.append(decoded[start_pos + j])
    
    return bytes(decoded)


data = b"banana"
encoded = lzss_encode(data)
decoded = lzss_decode(encoded)
print(f"LZSS: {decoded == data}")

LZSS: True


In [12]:
def lz78_encode(data: bytes, max_dict_size: int = 4096):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    # Словарь: фраза -> индекс
    dictionary = {b"": 0}
    next_index = 1
    
    current_phrase = b""
    
    while pos < n:
        # Добавляем следующий байт
        current_byte = bytes([data[pos]])
        new_phrase = current_phrase + current_byte
        
        if new_phrase in dictionary and next_index < max_dict_size:
            # Фраза есть в словаре, продолжаем
            current_phrase = new_phrase
            pos += 1
        else:
            # Выдаем пару (индекс префикса, текущий байт)
            prefix_index = dictionary[current_phrase] if current_phrase else 0
            
            # Кодируем индекс (2 байта) и символ (1 байт)
            encoded.extend(prefix_index.to_bytes(2, 'big'))
            encoded.append(data[pos])
            
            # Добавляем новую фразу в словарь (если не пустая и не превышен лимит)
            if new_phrase and next_index < max_dict_size:
                dictionary[new_phrase] = next_index
                next_index += 1
            
            # Сбрасываем текущую фразу
            current_phrase = b""
            pos += 1
    
    # Обработка остатка (если текущая фраза не пуста)
    if current_phrase:
        prefix_index = dictionary[current_phrase]
        encoded.extend(prefix_index.to_bytes(2, 'big'))
        encoded.append(0)
    
    return bytes(encoded)


def lz78_decode(encoded: bytes, max_dict_size: int = 4096):
    if not encoded:
        return b''
    
    decoded = bytearray()
    n = len(encoded)
    i = 0
    
    # Словарь: индекс -> фраза
    dictionary = {0: b""}
    next_index = 1
    
    while i + 2 < n:
        # Читаем индекс (2 байта) и символ (1 байт)
        index = int.from_bytes(encoded[i:i+2], 'big')
        next_byte = encoded[i+2]
        i += 3
        
        # Получаем фразу по индексу
        prefix = dictionary.get(index, b"")
        
        # Формируем новую фразу
        if next_byte == 0:
            new_phrase = prefix
        else:
            new_phrase = prefix + bytes([next_byte])
        
        # Добавляем в декодированный вывод
        decoded.extend(new_phrase)
        
        # Добавляем в словарь, если не превышен лимит
        if new_phrase and next_index < max_dict_size:
            dictionary[next_index] = new_phrase
            next_index += 1
    
    return bytes(decoded)

In [14]:
def lzw_encode(data: bytes, max_dict_size: int = 4096):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {bytes([i]): i for i in range(256)}
    next_index = 256
    
    current_phrase = b""
    
    while pos < n:
        # Добавляем следующий байт
        current_byte = bytes([data[pos]])
        new_phrase = current_phrase + current_byte
        
        if new_phrase in dictionary and next_index < max_dict_size:
            # Фраза есть в словаре, продолжаем
            current_phrase = new_phrase
            pos += 1
        else:
            # Выдаем индекс текущей фразы
            prefix_index = dictionary[current_phrase]
            encoded.extend(prefix_index.to_bytes(2, 'big'))
            
            # Добавляем новую фразу в словарь (если не превышен лимит)
            if new_phrase and next_index < max_dict_size:
                dictionary[new_phrase] = next_index
                next_index += 1
            
            # Сбрасываем текущую фразу
            current_phrase = current_byte
            pos += 1
    
    # Обработка остатка
    if current_phrase:
        prefix_index = dictionary[current_phrase]
        encoded.extend(prefix_index.to_bytes(2, 'big'))
    
    return bytes(encoded)


def lzw_decode(encoded: bytes, max_dict_size: int = 4096):
    if not encoded:
        return b''
    
    decoded = bytearray()
    n = len(encoded)
    i = 0
    
    # Инициализируем словарь всеми возможными байтами
    dictionary = {i: bytes([i]) for i in range(256)}
    next_index = 256
    
    # Читаем первый индекс
    if i + 2 > n:
        return b''
    
    prev_index = int.from_bytes(encoded[i:i+2], 'big')
    i += 2
    
    # Первая фраза
    prev_phrase = dictionary.get(prev_index, b"")
    decoded.extend(prev_phrase)
    
    while i + 2 <= n:
        # Читаем следующий индекс
        curr_index = int.from_bytes(encoded[i:i+2], 'big')
        i += 2
        
        # Получаем фразу
        if curr_index in dictionary:
            curr_phrase = dictionary[curr_index]
        elif curr_index == next_index and next_index < max_dict_size:
            # Специальный случай: фраза, которую только будем добавлять
            curr_phrase = prev_phrase + bytes([prev_phrase[0]])
        else:
            # Некорректный индекс
            break
        
        # Добавляем в декодированный вывод
        decoded.extend(curr_phrase)
        
        # Добавляем новую фразу в словарь
        if next_index < max_dict_size and prev_phrase:
            new_phrase = prev_phrase + bytes([curr_phrase[0]])
            dictionary[next_index] = new_phrase
            next_index += 1
        
        prev_phrase = curr_phrase
    
    return bytes(decoded)